## Init

In [1]:
import os
import numpy as np
import itertools as it
import random
import pandas as pd
import multiprocess as mp
import time
import sys
import matplotlib.pyplot as plt
import torch
import copy

from pyrqa.time_series import TimeSeries
from pyrqa.settings import Settings
from pyrqa.analysis_type import Classic
from pyrqa.neighbourhood import FixedRadius
from pyrqa.metric import EuclideanMetric
from pyrqa.computation import RQAComputation

pd.options.mode.copy_on_write = True

In [2]:
folder = "C:/Users/B00955739/Documents/Git/phd/Init/"

sys.path.append(folder)

import functions_v6_8 as fn
import ml_functions as mlf

In [3]:
folder = "C:/Users/B00955739/OneDrive - Ulster University/Documents/PhD/Results/Predicting_predictability/"
csv = "main_run_v2_corrected.csv"

In [4]:
out_name = "modelling_run_v1_1d"

In [5]:
full_df = pd.read_csv(folder + csv)

## Sampling

In [6]:
full_df_np = full_df[full_df.system_periodic == False].copy()

In [8]:
sample = full_df_np[full_df_np.dim == 1]
sample = sample[sample.system_id != 2]

In [9]:
sample.reverse = sample.reverse.astype(int)

sample.loc[sample.mat_type == "RAND", "epsilon"] = pd.NA
sample.loc[sample.mat_type != "OW", "reverse"] = pd.NA
sample.loc[sample.mat_type != "FCD", "gamma"] = pd.NA

In [10]:
sample.head(10)

,Unnamed: 0.1,Unnamed: 0,dim,mat_type,epsilon,map_list,map_list_str,gamma,reverse,matrix,...,lyap_3,lyap_4,lyap_5,lyap_6,lyap_7,lyap_8,lyap_9,largest_lyap,corrected,lyap_spectrum_full
0,0,0,1,NN,0.1,[<functions_v6.sincircle_map object at 0x00000...,"[('sin_circle_map', {'omega': 1.41421356237309...",NaN,NaN,[[1.]],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000016,False,"[1.5845574167039368e-05, nan, nan, nan, nan, n..."
1,1,1,1,NN,0.1,[<functions_v6.sincircle_map object at 0x00000...,"[('sin_circle_map', {'omega': 0.5, 'k': 2})]",NaN,NaN,[[1.]],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.351925,False,"[0.3519253487725863, nan, nan, nan, nan, nan, ..."
3,3,3,1,NN,0.1,[<functions_v6.tent_map object at 0x00000298DC...,"[('tent_map', {'a': 1.9999999999, 'b': 1})]",NaN,NaN,[[1.]],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.693147,False,"[0.6931471805096966, nan, nan, nan, nan, nan, ..."
4,4,4,1,NN,0.1,[<functions_v6.sluze_map object at 0x00000298E...,"[('sluze_map', {'m': 0.8, 'p': 0.2})]",NaN,NaN,[[1.]],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.417346,False,"[0.417345868882577, nan, nan, nan, nan, nan, n..."
5,5,5,1,NN,0.1,[<functions_v6.log_map object at 0x00000298E0A...,"[('log_map', {'a': 4})]",NaN,NaN,[[1.]],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.693071,False,"[0.6930713374415668, nan, nan, nan, nan, nan, ..."


## Modelling Params

In [11]:
n_layers = 4
layer_size = 200
lr = 0.001
batch_size = 32
lr_decay_factor = 0.1
epoch_vals = [100]

reps = 3

In [12]:
model_count = reps * len(sample)

print("{} models to train ({} systems; {} reps)".format(model_count, len(sample), reps))

15 models to train (5 systems; 3 reps)


In [13]:
update_at = np.unique(np.round(np.linspace(1, model_count, 100)))

In [14]:
models_complete = 0

result_list = []

for row_id, row in sample.iterrows():
    system_id = int(row['system_id'])
    sim_file = row['sim_file']
    modelling_len = int(row['modelling_len'])
    testing_len = int(row['testing_len'])
    n_tests = int(row['n_tests'])
    test_len = int(row['test_len'])
    embed_dim = int(row['embed_dim'])

    sim = np.load(sim_file)
    
    corrected_LDDP, corrected_LDDP_err = fn.calc_LDDP_corrected(sim)
    
    rqa_ts = TimeSeries(sim, embedding_dimension=embed_dim, time_delay=1)
    
    settings = Settings(rqa_ts)
    computation = RQAComputation.create(settings, verbose=False)
    result = computation.run()
    rqa_rr = result.recurrence_rate
    rqa_det = result.determinism
    
    assert len(sim) == modelling_len + testing_len
    modelling_series = sim[:modelling_len]
    testing_series = sim[-testing_len:]
    
    modelling_range = np.max(modelling_series) - np.min(modelling_series)
    sim_range = np.max(sim) - np.min(sim)
    
    assert len(modelling_series) == modelling_len
    assert len(testing_series) == testing_len
    test_dts = np.array(np.split(testing_series, n_tests))
    assert np.all(test_dts.shape == (n_tests, test_len))

    for rep_id in range(reps):
        
        result_dict = {
            "system_id": system_id,
            "embed_dim": embed_dim,
            "rep_id": rep_id,
            "modelling_range": modelling_range,
            "sim_range": sim_range,
            "corrected_LDDP": corrected_LDDP,
            "corrected_LDDP_err": corrected_LDDP_err,
            "RQA_rr": rqa_rr,
            "RQA_det": rqa_det
        }
        
        random_seed = int(10*system_id + rep_id)
        torch.manual_seed(random_seed)
        ##initialise model
        model = mlf.lstm(lstm_hs=layer_size, lstm_nl=n_layers)
        ## create dataloaders
        loader_dict = mlf.create_loaders(model_series=modelling_series, embed_dim=embed_dim, train_split=0.8,
                                         test_series=testing_series, batch_size=batch_size)
            
        trained_models, final_epoch, timings = mlf.train_model(model=model, train_loader=loader_dict["train_loader"],
                                        val_loader=loader_dict["val_loader"], epochs=epoch_vals, patience=100,
                                        loss_fn=torch.nn.L1Loss(), opt=torch.optim.Adam, start_lr=lr,
                                        lr_decay_factor=lr_decay_factor, if_save=False, save_folder=None, save_name=None,
                                        plot=False, verbose=False)

        result_dict["final_epoch"] = final_epoch

        for epoch_idx in range(len(trained_models)):
            final_dict = result_dict.copy()
            trained_model = trained_models[epoch_idx]
            final_dict["epoch"] = epoch_vals[epoch_idx]
            final_dict["model_time"] = timings[epoch_idx]
        
            final_dict["train_loss_mae"] = mlf.calc_loss(model=trained_model, dataloader=loader_dict["train_loader"], loss_fn=torch.nn.L1Loss())
            final_dict["val_loss_mae"] = mlf.calc_loss(model=trained_model, dataloader=loader_dict["val_loader"], loss_fn=torch.nn.L1Loss())
            final_dict.update(mlf.calc_point_metrics(model=trained_model, dataloader=loader_dict["test_loader"], prefix="test_loss"))
            
            pred_test_dts = mlf.pred_test_dts(test_dts=test_dts, model=trained_model, embed_dim=embed_dim)

            for t in [0.001, 0.01, 0.05]:
                pred_h = fn.calc_pred_h(test_dts=test_dts, pred_dts=pred_test_dts, embed_dim=embed_dim, thresh=t,
                                        percentiles=[1, 5, 10, 25, 50, 75, 90, 95, 99], if_plot=False)
                final_dict.update(pred_h)
            
            result_list.append(final_dict)
            
        models_complete += 1
        if models_complete in update_at:
            print("Completed training model {} of {} ({}%)".format(models_complete, model_count, np.round(100*(models_complete/model_count), 2)))

C:\Users\B00955739\AppData\Local\anaconda3\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
C:\Users\B00955739\AppData\Local\anaconda3\Lib\site-packages\numpy\core\_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


Completed training model 1 of 15 (6.67%)


C:\Users\B00955739\AppData\Local\anaconda3\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
C:\Users\B00955739\AppData\Local\anaconda3\Lib\site-packages\numpy\core\_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


Completed training model 2 of 15 (13.33%)


C:\Users\B00955739\AppData\Local\anaconda3\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
C:\Users\B00955739\AppData\Local\anaconda3\Lib\site-packages\numpy\core\_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


Completed training model 3 of 15 (20.0%)
Completed training model 4 of 15 (26.67%)
Completed training model 5 of 15 (33.33%)
Completed training model 6 of 15 (40.0%)
Completed training model 7 of 15 (46.67%)
Completed training model 8 of 15 (53.33%)
Completed training model 9 of 15 (60.0%)
Completed training model 10 of 15 (66.67%)
Completed training model 11 of 15 (73.33%)
Completed training model 12 of 15 (80.0%)
Completed training model 13 of 15 (86.67%)
Completed training model 14 of 15 (93.33%)
Completed training model 15 of 15 (100.0%)


In [15]:
res_df = pd.DataFrame(result_list)
res_df.head()

,system_id,embed_dim,rep_id,modelling_range,sim_range,corrected_LDDP,corrected_LDDP_err,RQA_rr,RQA_det,final_epoch,...,pred_h_p5 t=0.05,pred_h_p10 t=0.05,pred_h_p25 t=0.05,pred_h_p50 t=0.05,pred_h_p75 t=0.05,pred_h_p90 t=0.05,pred_h_p95 t=0.05,pred_h_p99 t=0.05,pred_h_sd t=0.05,pred_h_perfect_pred_count t=0.05
0,0,1,0,0.999957,0.999957,-0.117951,0.000000e+00,1.0,1.0,100,...,5.95,13.8,26.00,39.0,57.75,71.2,76.1,NaN,NaN,2
1,0,1,1,0.999957,0.999957,-0.117951,0.000000e+00,1.0,1.0,100,...,8.95,14.0,23.75,38.5,64.75,NaN,NaN,NaN,NaN,11
2,0,1,2,0.999957,0.999957,-0.117951,0.000000e+00,1.0,1.0,100,...,8.95,16.8,31.00,59.5,92.00,NaN,NaN,NaN,NaN,19
3,1,1,0,0.999628,0.999796,-0.579728,-1.110223e-16,1.0,1.0,100,...,0.00,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0
4,1,1,1,0.999628,0.999796,-0.579728,-1.110223e-16,1.0,1.0,100,...,0.00,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0


In [16]:
print("Saving CSV: {}".format(folder + out_name + ".csv"))

res_df.to_csv(folder + out_name + ".csv")

Saving CSV: C:/Users/B00955739/OneDrive - Ulster University/Documents/PhD/Results/Predicting_predictability/modelling_run_v1_1d.csv


In [17]:
print(len(res_df))

15
